# 🧳 AgentsVille Trip Planner

An AI travel assistant for the (fictional) city of **AgentsVille**, built in two stages:

1. **The Expert Planner** – an LLM acting as a travel planner turns the travelers' preferences,
   the weather forecast, and the list of available activities into a day-by-day itinerary.
   The itinerary is returned as JSON and validated against the `TravelPlan` Pydantic model.
2. **The Resourceful Assistant** – a **ReAct agent** (THINK → ACT → OBSERVE) revises the itinerary.
   It uses tools to look up activities, do arithmetic, and run evaluations, and it only finishes
   (via `final_answer_tool`) after the evaluations pass.

**How to read this notebook:** every section starts with a plain-English explanation of *what* it
does and *why*, then the code. Run the cells from top to bottom.

## 0. Setup

Imports, the OpenAI client, and a small helper to call the LLM.

* In the Udacity workspace the API key is a **Vocareum** key (it starts with `voc-`), and the
  client points at the Vocareum proxy. Put the key in an environment variable called
  `OPENAI_API_KEY` (or a `.env` file) — never paste secrets into a notebook you submit.
* `project_lib` is provided by Udacity. We only use its three simulated ("mocked") APIs.

In [ ]:
import ast
import datetime
import inspect
import json
import operator
import os
import re
from enum import Enum
from typing import Any, Callable, List, Optional

from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

from project_lib import (
    call_activities_api_mocked,
    call_activity_by_id_api_mocked,
    call_weather_api_mocked,
)

try:  # Optional: load OPENAI_API_KEY from a .env file if python-dotenv is installed.
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "voc-YOUR-KEY-HERE")
# Vocareum keys (voc-...) must go through the Vocareum proxy; normal OpenAI keys use the default URL.
OPENAI_BASE_URL = "https://openai.vocareum.com/v1" if OPENAI_API_KEY.startswith("voc-") else None

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)


class OpenAIModel(str, Enum):
    GPT_41 = "gpt-4.1"  # Strongest reasoning: used for planning and the ReAct agent.
    GPT_41_MINI = "gpt-4.1-mini"  # Cheaper: used for small yes/no evaluation calls.
    GPT_41_NANO = "gpt-4.1-nano"


MODEL = OpenAIModel.GPT_41
EVAL_MODEL = OpenAIModel.GPT_41_MINI


def do_chat_completion(messages: List[dict], model: str = MODEL, **kwargs) -> str:
    """Send a list of chat messages to the LLM and return the text of its reply."""
    response = client.chat.completions.create(
        model=model.value if isinstance(model, Enum) else model,
        messages=messages,
        temperature=kwargs.pop("temperature", 0.1),  # Low temperature = more consistent output.
        **kwargs,
    )
    return response.choices[0].message.content or ""


def print_in_box(text: str, title: str = "") -> None:
    """Pretty-print text inside a simple box so long outputs are easy to read."""
    width = 100
    print("╔" + f" {title} ".center(width, "═") + "╗")
    for line in str(text).splitlines() or [""]:
        while len(line) > width - 2:
            print("║ " + line[: width - 2] + " ║")
            line = line[width - 2 :]
        print("║ " + line.ljust(width - 2) + " ║")
    print("╚" + "═" * width + "╝")

## 1. Define the vacation details (`VacationInfo`)

The travelers' details arrive as plain JSON. We define **Pydantic models** for them. A Pydantic
model is a blueprint that *checks* data: if a date is malformed, an interest is misspelled, or a
field is missing, Pydantic raises an error immediately instead of letting bad data reach the LLM.

* `Interest` – the fixed list of interests AgentsVille activities are tagged with.
* `Traveler` – one person: name, age, interests.
* `VacationInfo` – the whole trip: travelers, destination, arrival/departure dates, budget.

In [ ]:
class Interest(str, Enum):
    ART = "art"
    COOKING = "cooking"
    COMEDY = "comedy"
    DANCING = "dancing"
    FITNESS = "fitness"
    GARDENING = "gardening"
    HIKING = "hiking"
    MOVIES = "movies"
    MUSIC = "music"
    PHOTOGRAPHY = "photography"
    READING = "reading"
    SPORTS = "sports"
    TECHNOLOGY = "technology"
    THEATRE = "theatre"
    TENNIS = "tennis"
    WRITING = "writing"


VACATION_INFO_DICT = {
    "travelers": [
        {"name": "Yuri", "age": 30, "interests": ["tennis", "cooking", "comedy", "technology"]},
        {"name": "Hiro", "age": 25, "interests": ["reading", "music", "theatre", "art"]},
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",
    "date_of_departure": "2025-06-12",
    "budget": 130,
}


class Traveler(BaseModel):
    """A single traveler and the things they enjoy."""

    name: str = Field(description="The traveler's first name.")
    age: int = Field(ge=0, description="The traveler's age in years.")
    interests: List[Interest] = Field(description="The traveler's interests.")


class VacationInfo(BaseModel):
    """Everything we know about the trip before planning starts."""

    travelers: List[Traveler] = Field(min_length=1, description="Everyone going on the trip.")
    destination: str = Field(description="The city being visited.")
    date_of_arrival: datetime.date = Field(description="First day of the trip (YYYY-MM-DD).")
    date_of_departure: datetime.date = Field(description="Last day of the trip (YYYY-MM-DD).")
    budget: int = Field(ge=0, description="Maximum total spend on activities for the whole group.")


vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)

assert vacation_info.date_of_arrival <= vacation_info.date_of_departure, "Arrival must not be after departure."
print_in_box(vacation_info.model_dump_json(indent=2), "Vacation Info")

## 2. Gather the "external" data: weather and activities

We read the **start and end dates from the `VacationInfo` object**, walk through every day of the
trip (inclusive), and call the simulated weather and activities APIs for each date.

In [ ]:
def get_trip_dates(info: VacationInfo) -> List[str]:
    """Return every date of the trip, from arrival to departure inclusive, as 'YYYY-MM-DD' strings."""
    number_of_days = (info.date_of_departure - info.date_of_arrival).days + 1
    return [
        (info.date_of_arrival + datetime.timedelta(days=offset)).isoformat()
        for offset in range(number_of_days)
    ]


trip_dates = get_trip_dates(vacation_info)

weather_for_dates = [
    call_weather_api_mocked(date=date, city=vacation_info.destination) for date in trip_dates
]

activities_for_dates = [
    activity
    for date in trip_dates
    for activity in call_activities_api_mocked(date=date, city=vacation_info.destination)
]

print(f"Trip dates: {trip_dates}")
print_in_box(json.dumps(weather_for_dates, indent=2), "Weather")
print(f"Found {len(activities_for_dates)} activities across {len(trip_dates)} days.")

## 3. Define the itinerary structure (`TravelPlan`)

The LLM must return an itinerary that fits this exact shape. The `description` on each field is
included in the JSON schema we give the LLM, so the descriptions double as instructions.

In [ ]:
class Weather(BaseModel):
    temperature: float = Field(description="Forecast temperature.")
    temperature_unit: str = Field(description="Unit of the temperature, e.g. 'celsius'.")
    condition: str = Field(description="Short weather condition, e.g. 'sunny' or 'thunderstorm'.")


class Activity(BaseModel):
    activity_id: str = Field(description="Exact activity_id from the activities API. Never invent one.")
    name: str
    start_time: str = Field(description="Start time exactly as given by the API, e.g. '2025-06-10 09:00'.")
    end_time: str = Field(description="End time exactly as given by the API, e.g. '2025-06-10 11:00'.")
    location: str
    description: str
    price: int = Field(description="Price exactly as given by the API.")
    related_interests: List[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity = Field(description="The activity, copied exactly from the activities API.")
    reasons_for_recommendation: List[str] = Field(
        description="Why this activity fits the travelers: interests, weather, budget, timing."
    )


class ItineraryDay(BaseModel):
    date: datetime.date = Field(description="The date of this itinerary day (YYYY-MM-DD).")
    weather: Weather = Field(description="The weather forecast for this date.")
    activity_recommendations: List[ActivityRecommendation] = Field(
        description="Activities for this day, in chronological order, with no overlapping times."
    )


class TravelPlan(BaseModel):
    city: str = Field(description="The destination city.")
    start_date: datetime.date = Field(description="Must equal the date of arrival.")
    end_date: datetime.date = Field(description="Must equal the date of departure.")
    total_cost: int = Field(description="Sum of the prices of every recommended activity.")
    itinerary_days: List[ItineraryDay] = Field(description="One entry per day of the trip, in order.")


TRAVEL_PLAN_SCHEMA = json.dumps(TravelPlan.model_json_schema(), indent=2)

## 4. Stage 1 — The Expert Planner (`ITINERARY_AGENT_SYSTEM_PROMPT`)

The system prompt below:

* **Assigns a role** – an expert AgentsVille travel planner.
* **Guides step-by-step (Chain-of-Thought) reasoning** – a numbered planning process the model
  follows for every day, written out in an `## Analysis` section before the JSON.
* **Specifies the output format** – a single JSON block matching the `TravelPlan` schema, plus a
  short example day.
* **Provides all context** – the `VacationInfo`, the weather, and the activities.

In [ ]:
ITINERARY_AGENT_SYSTEM_PROMPT = f"""
## ROLE
You are an expert travel planner and concierge for the city of {vacation_info.destination}.
You build detailed, realistic, day-by-day itineraries tailored to each traveler's interests,
the weather forecast, and the group's budget.

## TASK
Create a complete itinerary covering every day from {vacation_info.date_of_arrival} to
{vacation_info.date_of_departure} (inclusive), using ONLY the activities listed in the CONTEXT section.

## PLANNING PROCESS (think step by step)
Before writing the final JSON, reason through the plan under a heading `## Analysis`:

1. **Understand the travelers.** List each traveler and their interests. Note the total budget
   ({vacation_info.budget}), which covers the WHOLE group across the WHOLE trip.
2. **Review each day's weather.** For each date, note the condition and whether outdoor activities
   are safe and enjoyable (e.g. rain, thunderstorms, snow, extreme heat or cold rule out outdoor-only
   activities).
3. **Shortlist activities per day.** For each date, list the available activities and mark each as:
   - interest match: which traveler(s) it suits,
   - weather fit: compatible / not compatible with that day's forecast,
   - price.
   Discard activities that are incompatible with the weather.
4. **Build each day.** Pick activities that:
   - cover every traveler's interests at least once during the trip,
   - do not overlap in time (check start and end times),
   - are in chronological order.
   Aim for at least two activities per day when the budget allows.
5. **Check the budget.** Add up the prices of every chosen activity step by step. If the total
   exceeds {vacation_info.budget}, swap expensive activities for cheaper ones and recalculate.
6. **Final review.** Confirm every date is covered, start_date/end_date match the trip dates,
   every activity is copied exactly from the CONTEXT, and total_cost equals the sum of prices.

## RULES
- Use ONLY activities from the CONTEXT. Copy every activity field EXACTLY (activity_id, name,
  start_time, end_time, location, description, price, related_interests). Never invent activities.
- Copy each day's weather (temperature, temperature_unit, condition) from the weather data.
- `total_cost` must equal the exact sum of `price` across all recommended activities and must not
  exceed {vacation_info.budget}.
- Each recommendation must include specific `reasons_for_recommendation` (which traveler's interest
  it matches, why the weather suits it, how it fits the budget/schedule).

## OUTPUT FORMAT
Respond with two sections, in this order:

## Analysis
<your step-by-step reasoning following the PLANNING PROCESS>

## Final Output
```json
<a single JSON object that is a valid TravelPlan>
```

The JSON MUST validate against this TravelPlan JSON schema:
```json
{TRAVEL_PLAN_SCHEMA}
```

Example of ONE itinerary day inside `itinerary_days` (illustrative values only):
```json
{{
  "date": "2025-01-01",
  "weather": {{"temperature": 20, "temperature_unit": "celsius", "condition": "sunny"}},
  "activity_recommendations": [
    {{
      "activity": {{
        "activity_id": "event-2025-01-01-0",
        "name": "Example Open-Air Concert",
        "start_time": "2025-01-01 18:00",
        "end_time": "2025-01-01 20:00",
        "location": "Example Park (outdoor)",
        "description": "Live music outdoors.",
        "price": 20,
        "related_interests": ["music"]
      }},
      "reasons_for_recommendation": [
        "Matches Hiro's interest in music.",
        "Sunny weather is ideal for an outdoor concert.",
        "At 20 it leaves room in the budget for other activities."
      ]
    }}
  ]
}}
```

## CONTEXT

### Vacation Info
```json
{vacation_info.model_dump_json(indent=2)}
```

### Weather Forecast (one entry per date)
```json
{json.dumps(weather_for_dates, indent=2)}
```

### Available Activities
```json
{json.dumps(activities_for_dates, indent=2)}
```
""".strip()

### Helper: pull the JSON out of the LLM's reply

The model writes its reasoning first and the JSON last. This helper finds the JSON so Pydantic can
validate it.

In [ ]:
def extract_json_from_text(text: str) -> str:
    """Return the last ```json ... ``` block in `text` (or the outermost {...} if there is no block)."""
    blocks = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if blocks:
        return blocks[-1]
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in the LLM response.")
    return text[start : end + 1]

### Generate the initial itinerary

If the JSON doesn't validate, we send the validation error back to the model and ask it to fix the
output (up to 3 attempts). This makes the pipeline robust to the occasional formatting slip.

In [ ]:
def generate_travel_plan(system_prompt: str, max_attempts: int = 3) -> TravelPlan:
    """Ask the planner LLM for a TravelPlan, retrying with the error message if validation fails."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Please create the itinerary for our trip."},
    ]
    for attempt in range(1, max_attempts + 1):
        response = do_chat_completion(messages, model=MODEL)
        try:
            return TravelPlan.model_validate_json(extract_json_from_text(response))
        except (ValueError, ValidationError) as error:
            print(f"Attempt {attempt}: the itinerary did not validate. Asking the model to fix it.")
            messages += [
                {"role": "assistant", "content": response},
                {"role": "user", "content": f"The JSON was invalid:\n{error}\nPlease return a corrected TravelPlan."},
            ]
    raise RuntimeError(f"Could not produce a valid TravelPlan after {max_attempts} attempts.")


def print_travel_plan(plan: TravelPlan) -> None:
    """Show a TravelPlan as a readable, day-by-day schedule."""
    lines = [f"{plan.city}: {plan.start_date} → {plan.end_date}   Total cost: {plan.total_cost}", ""]
    for day in plan.itinerary_days:
        w = day.weather
        lines.append(f"📅 {day.date}  ({w.condition}, {w.temperature:g}° {w.temperature_unit})")
        for rec in day.activity_recommendations:
            a = rec.activity
            lines.append(f"   • {a.start_time[-5:]}–{a.end_time[-5:]}  {a.name}  [{a.price}]  @ {a.location}")
            for reason in rec.reasons_for_recommendation:
                lines.append(f"       - {reason}")
        lines.append("")
    print_in_box("\n".join(lines), "Travel Plan")


travel_plan_1 = generate_travel_plan(ITINERARY_AGENT_SYSTEM_PROMPT)
print_travel_plan(travel_plan_1)

## 5. Evaluating the itinerary

LLMs can make mistakes (wrong totals, invented events, a picnic in a thunderstorm), so we check the
plan with **evaluation functions**. Each one raises an `AgentError` describing the problem, which the
ReAct agent can later read and fix.

| Evaluation | What it checks | How |
|---|---|---|
| `eval_start_end_dates_match` | Plan covers the right dates | Python |
| `eval_total_cost_is_accurate` | `total_cost` = sum of prices | Python |
| `eval_total_cost_is_within_budget` | Total ≤ budget | Python |
| `eval_itinerary_events_match_actual_events` | No invented / altered activities | Python + API |
| `eval_itinerary_satisfies_interests` | Every traveler gets ≥1 matching activity | Python |
| `eval_activities_and_weather_are_compatible` | No outdoor-only event in bad weather | **LLM** |

In [ ]:
class AgentError(Exception):
    """Raised by an evaluation function when the itinerary breaks a rule."""


class EvaluationResults(BaseModel):
    success: bool
    failures: List[str]
    eval_functions: List[str]


def get_eval_results(info: VacationInfo, plan: TravelPlan, eval_functions: List[Callable]) -> EvaluationResults:
    """Run every evaluation function and collect the failures instead of stopping at the first one."""
    failures = []
    for eval_fn in eval_functions:
        try:
            eval_fn(info, plan)
        except AgentError as error:
            failures.append(f"{eval_fn.__name__}: {error}")
    return EvaluationResults(
        success=not failures,
        failures=failures,
        eval_functions=[fn.__name__ for fn in eval_functions],
    )


def eval_start_end_dates_match(info: VacationInfo, plan: TravelPlan) -> None:
    if plan.start_date != info.date_of_arrival or plan.end_date != info.date_of_departure:
        raise AgentError(
            f"Plan dates {plan.start_date}–{plan.end_date} do not match the trip "
            f"{info.date_of_arrival}–{info.date_of_departure}."
        )
    planned_dates = [day.date.isoformat() for day in plan.itinerary_days]
    if planned_dates != get_trip_dates(info):
        raise AgentError(f"itinerary_days must cover {get_trip_dates(info)} in order, got {planned_dates}.")


def eval_total_cost_is_accurate(info: VacationInfo, plan: TravelPlan) -> None:
    actual = sum(rec.activity.price for day in plan.itinerary_days for rec in day.activity_recommendations)
    if plan.total_cost != actual:
        raise AgentError(f"total_cost is {plan.total_cost}, but the activity prices add up to {actual}.")


def eval_total_cost_is_within_budget(info: VacationInfo, plan: TravelPlan) -> None:
    if plan.total_cost > info.budget:
        raise AgentError(f"total_cost {plan.total_cost} exceeds the budget of {info.budget}.")


def eval_itinerary_events_match_actual_events(info: VacationInfo, plan: TravelPlan) -> None:
    problems = []
    for day in plan.itinerary_days:
        for rec in day.activity_recommendations:
            planned = rec.activity
            real = call_activity_by_id_api_mocked(planned.activity_id)
            if real is None:
                problems.append(f"'{planned.activity_id}' ({planned.name}) does not exist.")
                continue
            for field in ("name", "start_time", "end_time", "location", "price"):
                if str(getattr(planned, field)) != str(real[field]):
                    problems.append(f"'{planned.activity_id}' field '{field}' is '{getattr(planned, field)}' but should be '{real[field]}'.")
            if not real["start_time"].startswith(day.date.isoformat()):
                problems.append(f"'{planned.activity_id}' is on {real['start_time'][:10]}, not {day.date}.")
    if problems:
        raise AgentError("Itinerary contains activities that don't match the API: " + " ".join(problems))


def eval_itinerary_satisfies_interests(info: VacationInfo, plan: TravelPlan) -> None:
    planned_interests = {
        interest
        for day in plan.itinerary_days
        for rec in day.activity_recommendations
        for interest in rec.activity.related_interests
    }
    unhappy = [t.name for t in info.travelers if not planned_interests.intersection(t.interests)]
    if unhappy:
        raise AgentError(f"No activity matches the interests of: {', '.join(unhappy)}.")

### Weather compatibility check (`ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT`)

Deciding whether "Shakespeare in the Park" works in a thunderstorm needs common sense, so we ask
an LLM. The prompt defines the **role and task**, the **exact output format**, and several
**worked examples** so the answers are consistent and easy to parse.

In [ ]:
ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
## ROLE
You are a meticulous event-safety and weather analyst for a travel agency.

## TASK
You will receive ONE activity and the weather forecast for the day it takes place.
Decide whether the activity should be AVOIDED because of the weather.

## HOW TO DECIDE
1. Determine whether the activity is indoor, outdoor, or unclear, using the name, location and
   description (words like "park", "trail", "outdoor", "open-air", "courts", "amphitheater" suggest
   outdoor; "museum", "café", "club", "hub", "indoor" suggest indoor).
2. Check whether the description mentions an indoor backup or weather-proof venue.
3. Judge the weather:
   - Unsafe/unpleasant for outdoor activities: thunderstorm, heavy rain, rain, snow, hail,
     strong wind, extreme heat or cold.
   - Fine for outdoor activities: sunny, clear, partly cloudy, cloudy, mild temperatures.
4. Decide:
   - Indoor activity → IS_COMPATIBLE (weather does not matter).
   - Outdoor activity with fine weather → IS_COMPATIBLE.
   - Outdoor activity with unsafe/unpleasant weather and no indoor backup → IS_INCOMPATIBLE.
   - Not enough information to be sure → IS_COMPATIBLE (give the benefit of the doubt).

## OUTPUT FORMAT (follow exactly)
REASONING: <one or two sentences explaining your decision>
FINAL ANSWER: <IS_COMPATIBLE or IS_INCOMPATIBLE>

Output nothing else. The last line must be exactly "FINAL ANSWER: IS_COMPATIBLE" or
"FINAL ANSWER: IS_INCOMPATIBLE".

## EXAMPLES

Example 1
Activity: "Sunrise Summit Hike" at "Mount Prompt Trailhead (outdoor)" - guided hike on exposed ridgelines.
Weather: thunderstorm, 18 celsius
REASONING: The hike is fully outdoors on exposed terrain, and lightning makes this dangerous.
FINAL ANSWER: IS_INCOMPATIBLE

Example 2
Activity: "Gallery of Generative Art" at "AgentsVille Modern Art Museum (indoor)".
Weather: heavy rain, 15 celsius
REASONING: The exhibition is inside a museum, so rain does not affect it.
FINAL ANSWER: IS_COMPATIBLE

Example 3
Activity: "Symphony Under the Stars" at "AgentsVille Amphitheater (outdoor)".
Weather: sunny, 25 celsius
REASONING: It is an outdoor concert, and clear, warm weather is ideal for it.
FINAL ANSWER: IS_COMPATIBLE

Example 4
Activity: "Street Food Night Market" at "Innovation Plaza (open-air)" - moves indoors for rain.
Weather: rain, 17 celsius
REASONING: The market is open-air but has an indoor backup for rain, so it can still go ahead.
FINAL ANSWER: IS_COMPATIBLE

Example 5
Activity: "Beach Volleyball Tournament" at "Sunny Shore Beach".
Weather: snow, -2 celsius
REASONING: Beach volleyball is played outdoors and snow with freezing temperatures makes it impractical.
FINAL ANSWER: IS_INCOMPATIBLE
""".strip()


def eval_activities_and_weather_are_compatible(info: VacationInfo, plan: TravelPlan) -> None:
    incompatible = []
    for day in plan.itinerary_days:
        for rec in day.activity_recommendations:
            activity = rec.activity
            messages = [
                {"role": "system", "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Activity: {activity.model_dump_json()}\n"
                    f"Weather: {day.weather.model_dump_json()}"
                )},
            ]
            answer = do_chat_completion(messages, model=EVAL_MODEL)
            if "IS_INCOMPATIBLE" in answer.split("FINAL ANSWER:")[-1]:
                incompatible.append(f"'{activity.name}' ({activity.activity_id}) on {day.date} [{day.weather.condition}]")
    if incompatible:
        raise AgentError(
            "These activities are incompatible with the weather and should be replaced: "
            + "; ".join(incompatible)
        )


ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_total_cost_is_within_budget,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_activities_and_weather_are_compatible,
]

eval_results_1 = get_eval_results(vacation_info, travel_plan_1, ALL_EVAL_FUNCTIONS)
print_in_box(eval_results_1.model_dump_json(indent=2), "Evaluation of the initial itinerary")

## 6. Traveler feedback

After seeing the first draft, the travelers ask for a change. We add an extra (LLM-judged)
evaluation that checks the feedback has been applied. The ReAct agent must make this pass too.

In [ ]:
TRAVELER_FEEDBACK = "I want to have at least two activities per day."

TRAVELER_FEEDBACK_IS_INCORPORATED_SYSTEM_PROMPT = """
## ROLE
You are a strict quality reviewer for a travel agency.

## TASK
You will receive traveler feedback and a travel itinerary as JSON. Decide whether the itinerary
fully satisfies the feedback. Check every day of the itinerary.

## OUTPUT FORMAT (follow exactly)
REASONING: <short explanation, citing specific days if the feedback is not met>
FINAL ANSWER: <IS_INCORPORATED or IS_NOT_INCORPORATED>
""".strip()


def eval_traveler_feedback_is_incorporated(info: VacationInfo, plan: TravelPlan) -> None:
    messages = [
        {"role": "system", "content": TRAVELER_FEEDBACK_IS_INCORPORATED_SYSTEM_PROMPT},
        {"role": "user", "content": f"Traveler feedback: {TRAVELER_FEEDBACK}\n\nItinerary:\n{plan.model_dump_json(indent=2)}"},
    ]
    answer = do_chat_completion(messages, model=EVAL_MODEL)
    if "IS_NOT_INCORPORATED" in answer.split("FINAL ANSWER:")[-1]:
        raise AgentError(f"Traveler feedback not incorporated. Reviewer said: {answer.strip()}")


ALL_EVAL_FUNCTIONS.append(eval_traveler_feedback_is_incorporated)

## 7. Tools for the ReAct agent

A **tool** is a normal Python function the agent can ask us to run. The agent only sees each tool's
**name, parameters and docstring**, so the docstrings are written as instructions for the LLM:
what the tool does, when to use it, each parameter's type/format, and what it returns.

In [ ]:
_SAFE_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.FloorDiv: operator.floordiv, ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _safe_eval(node: ast.AST) -> float:
    """Evaluate a parsed arithmetic expression without using Python's unsafe eval()."""
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPERATORS:
        return _SAFE_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPERATORS:
        return _SAFE_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Only numbers and + - * / % ** // ( ) are allowed.")


def calculator_tool(input_expression: str) -> float:
    """Evaluate an arithmetic expression and return the numeric result.

    Use this whenever you need exact arithmetic, especially to add up activity prices for
    `total_cost` or to check the total against the budget. Do not do mental math.

    Args:
        input_expression (str): An arithmetic expression using numbers, + - * / % ** // and
            parentheses. Example: "15 + 5 + 20 + 12".

    Returns:
        float: The result of the expression, e.g. 52.0.
    """
    return float(_safe_eval(ast.parse(input_expression, mode="eval")))


def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """Look up every activity available in a city on a specific date.

    Use this tool when you need to find activities for a given day, for example to replace an
    activity that is incompatible with the weather, to add more activities to a day, or to find
    cheaper or interest-matching options. Only activities returned by this tool may appear in the
    itinerary, and their fields must be copied exactly.

    Args:
        date (str): The date to search, in ISO format "YYYY-MM-DD" (e.g. "2025-06-10").
        city (str): The city name exactly as written in the vacation info (e.g. "AgentsVille").

    Returns:
        list[dict]: A list of activities (possibly empty). Each activity has the keys:
            activity_id (str), name (str), start_time (str, "YYYY-MM-DD HH:MM"),
            end_time (str, "YYYY-MM-DD HH:MM"), location (str), description (str),
            price (int), related_interests (list[str]).
    """
    datetime.date.fromisoformat(date)  # Raises a clear error if the date format is wrong.
    return call_activities_api_mocked(date=date, city=city)


def run_evals_tool(travel_plan: TravelPlan) -> dict:
    """Run all evaluation checks on a proposed travel plan and report any failures.

    You MUST call this tool on your revised plan before calling final_answer_tool. If any check
    fails, fix the plan and run this tool again until "success" is true.

    Args:
        travel_plan (TravelPlan): The COMPLETE revised travel plan as a JSON object that matches
            the TravelPlan schema.

    Returns:
        dict: {"success": bool, "failures": list[str], "eval_functions": list[str]}.
    """
    if isinstance(travel_plan, dict):
        travel_plan = TravelPlan.model_validate(travel_plan)
    return get_eval_results(vacation_info, travel_plan, ALL_EVAL_FUNCTIONS).model_dump()


def final_answer_tool(final_output: TravelPlan) -> TravelPlan:
    """Submit the final, revised travel plan and end the task.

    Only call this after run_evals_tool has returned "success": true for this exact plan.
    This must be your last action.

    Args:
        final_output (TravelPlan): The COMPLETE final travel plan as a JSON object that matches
            the TravelPlan schema.

    Returns:
        TravelPlan: The final travel plan.
    """
    if isinstance(final_output, dict):
        final_output = TravelPlan.model_validate(final_output)
    return final_output


ALL_TOOLS = [calculator_tool, get_activities_by_date_tool, run_evals_tool, final_answer_tool]


def get_tool_descriptions_string(tools: List[Callable]) -> str:
    """Build the tool list for the system prompt from each function's name, signature and docstring."""
    descriptions = []
    for tool in tools:
        signature = str(inspect.signature(tool)).replace("__main__.", "")
        descriptions.append(f"### {tool.__name__}{signature}\n{inspect.getdoc(tool)}")
    return "\n\n".join(descriptions)


print(get_tool_descriptions_string(ALL_TOOLS))

## 8. Stage 2 — The Resourceful Assistant (`ITINERARY_REVISION_AGENT_SYSTEM_PROMPT`)

This prompt turns the LLM into a **ReAct agent**. It:

* States the **role** (itinerary revision specialist) and the **task**.
* Explains the **THINK → ACT → OBSERVE** cycle explicitly.
* **Lists every tool** with its purpose and parameters (generated automatically from the docstrings).
* Gives the **exact ACTION format**: `{"tool_name": "[tool_name]", "arguments": {"arg1": "value1", ...}}`.
* Requires **`run_evals_tool` before `final_answer_tool`**, and says the task ends by calling
  **`final_answer_tool`**.
* Includes the **`TravelPlan` schema** so the final plan has the right structure.

In [ ]:
ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
## ROLE
You are an expert itinerary revision specialist for {vacation_info.destination}. You improve an
existing travel plan so that it satisfies the travelers' feedback and passes every quality check.

## TASK
You will receive the current travel plan and the travelers' feedback. Revise the plan so that:
- the travelers' feedback is fully incorporated,
- every activity is real (copied exactly from get_activities_by_date_tool results),
- no activity is incompatible with that day's weather,
- every traveler's interests are covered,
- total_cost is the exact sum of activity prices and stays within the budget of {vacation_info.budget},
- the plan covers every day from {vacation_info.date_of_arrival} to {vacation_info.date_of_departure}.

## HOW YOU WORK: THE THINK-ACT-OBSERVE CYCLE
You solve the task in a loop. In EVERY response you do exactly two things:

1. **THINK** – Reason step by step about the current situation: what the feedback and the last
   observation tell you, what is still missing or wrong, and which single tool call will help most.
2. **ACT** – Call exactly ONE tool using the ACTION format below. Then STOP and wait.

After each ACTION, the system runs the tool and replies with:

3. **OBSERVATION** – The tool's result (or an error message). Use it in your next THINK step.

Repeat THINK → ACT → OBSERVE until the plan passes all evaluations, then finish with final_answer_tool.

## AVAILABLE TOOLS
{get_tool_descriptions_string(ALL_TOOLS)}

## ACTION FORMAT (follow exactly)
Each response must look like this:

THINK:
<your reasoning>

ACTION:
{{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", ...}}}}

Rules for the ACTION:
- It must be a single valid JSON object on its own after the line "ACTION:" (no code fences,
  no comments, no text after it).
- "tool_name" must be one of: {", ".join(tool.__name__ for tool in ALL_TOOLS)}.
- "arguments" must use the exact parameter names listed for that tool.
- Only ONE action per response. Never write an OBSERVATION yourself.

Examples:
ACTION:
{{"tool_name": "get_activities_by_date_tool", "arguments": {{"date": "2025-06-10", "city": "AgentsVille"}}}}

ACTION:
{{"tool_name": "calculator_tool", "arguments": {{"input_expression": "15 + 5 + 20"}}}}

ACTION:
{{"tool_name": "run_evals_tool", "arguments": {{"travel_plan": {{...complete TravelPlan JSON...}}}}}}

ACTION:
{{"tool_name": "final_answer_tool", "arguments": {{"final_output": {{...complete TravelPlan JSON...}}}}}}

## RECOMMENDED STRATEGY
1. Read the feedback and the current plan; identify every change needed.
2. Call get_activities_by_date_tool for each date you need to change, to see the real options.
3. Choose activities that fit the feedback, weather, interests, timing (no overlaps) and budget.
4. Use calculator_tool to compute total_cost exactly.
5. Call run_evals_tool with the COMPLETE revised plan.
6. If any evaluation fails, read the failures, fix the plan, and call run_evals_tool again.
7. When run_evals_tool returns "success": true, call final_answer_tool with that exact plan.

## EXIT INSTRUCTIONS (mandatory)
- You MUST call run_evals_tool on your revised plan BEFORE calling final_answer_tool.
- Only call final_answer_tool after run_evals_tool returns "success": true for that same plan.
- The task ends ONLY when you call final_answer_tool with the complete final TravelPlan as the
  "final_output" argument. Never end with plain text.

## TRAVEL PLAN SCHEMA
Every travel plan you pass to run_evals_tool or final_answer_tool must be a complete JSON object
that validates against this TravelPlan JSON schema:
```json
{TRAVEL_PLAN_SCHEMA}
```

## VACATION INFO
```json
{vacation_info.model_dump_json(indent=2)}
```
""".strip()

### The ReAct loop

This is the Python "engine" around the agent. Each turn it:

1. Sends the conversation to the LLM (the agent THINKs and writes an ACTION).
2. Parses the ACTION JSON and runs the matching Python tool (ACT).
3. Sends the result back as an `OBSERVATION:` message.

It stops when the agent calls `final_answer_tool`. As a safety net, the code also refuses a final
answer if `run_evals_tool` hasn't been called yet, and stops after `max_steps` turns.

In [ ]:
TRAVEL_PLAN_ARGUMENTS = {"travel_plan", "final_output"}  # Tool arguments that must be a TravelPlan.


def parse_action(response: str) -> dict:
    """Find the JSON object after 'ACTION:' in the agent's response."""
    if "ACTION:" not in response:
        raise ValueError("No 'ACTION:' found. Respond with THINK: ... then ACTION: {json}.")
    after_action = response.split("ACTION:")[-1]
    after_action = re.sub(r"^\s*```(?:json)?", "", after_action.strip())
    start = after_action.find("{")
    if start == -1:
        raise ValueError("No JSON object found after 'ACTION:'.")
    action, _ = json.JSONDecoder().raw_decode(after_action[start:])
    if not isinstance(action, dict) or "tool_name" not in action or "arguments" not in action:
        raise ValueError('ACTION must look like {"tool_name": "...", "arguments": {...}}.')
    return action


def run_react_agent(messages: List[dict], tools: List[Callable], max_steps: int = 15) -> TravelPlan:
    """Run the THINK-ACT-OBSERVE loop until the agent calls final_answer_tool."""
    tools_by_name = {tool.__name__: tool for tool in tools}
    evals_have_run = False

    for step in range(1, max_steps + 1):
        response = do_chat_completion(messages, model=MODEL)
        messages.append({"role": "assistant", "content": response})
        print_in_box(response, f"Step {step}: agent")

        try:
            action = parse_action(response)
            tool_name, arguments = action["tool_name"], action["arguments"]
            if tool_name not in tools_by_name:
                raise ValueError(f"Unknown tool '{tool_name}'. Choose one of {list(tools_by_name)}.")

            # Convert travel-plan arguments from plain JSON into validated TravelPlan objects.
            for name in TRAVEL_PLAN_ARGUMENTS & arguments.keys():
                arguments[name] = TravelPlan.model_validate(arguments[name])

            if tool_name == "final_answer_tool":
                if not evals_have_run:
                    raise ValueError("You must call run_evals_tool before final_answer_tool.")
                return final_answer_tool(**arguments)

            result = tools_by_name[tool_name](**arguments)
            if tool_name == "run_evals_tool":
                evals_have_run = True
            observation = json.dumps(result, indent=2, default=str)
        except Exception as error:  # Any problem is reported back so the agent can correct itself.
            observation = f"ERROR: {type(error).__name__}: {error}"

        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
        print_in_box(observation, f"Step {step}: observation")

    raise RuntimeError(f"The agent did not call final_answer_tool within {max_steps} steps.")

### Run the agent

In [ ]:
revision_messages = [
    {"role": "system", "content": ITINERARY_REVISION_AGENT_SYSTEM_PROMPT},
    {"role": "user", "content": (
        f"Here is the current travel plan:\n{travel_plan_1.model_dump_json(indent=2)}\n\n"
        f"Traveler feedback: {TRAVELER_FEEDBACK}\n\n"
        "Please revise the itinerary."
    )},
]

travel_plan_2 = run_react_agent(revision_messages, ALL_TOOLS)
print_travel_plan(travel_plan_2)

## 9. Final check

We run every evaluation one last time, independently of the agent, to confirm the final plan is
correct.

In [ ]:
final_eval_results = get_eval_results(vacation_info, travel_plan_2, ALL_EVAL_FUNCTIONS)
print_in_box(final_eval_results.model_dump_json(indent=2), "Evaluation of the final itinerary")
assert final_eval_results.success, "The final itinerary failed at least one evaluation."
print("🎉 The final itinerary passes every evaluation!")